# Tech Challenge — Fase 4
## Notebook 05 — Avaliação do RAG — Execução Databricks

**Projeto:** Voice of Customer Intelligence com RAG — Olist Brazilian E-Commerce.

### Objetivo
Avaliar a recuperação, a qualidade das respostas, as fontes, as abstenções e o desempenho do pipeline
no Databricks concluído no Notebook 04. Mantemos sua dinâmica de objetivo, justificativa, ação e interpretação.

### Ponto de partida
O 03 no Databricks gerou documentos, vetores, coleção Chroma e src/busca_vetorial.py.
O 04 no Databricks gerou src/rag_pipeline_no Databricks.py. Execute inclusive sua última célula de persistência do módulo.
Este notebook importa esses arquivos, sem reescrever a recuperação, os prompts ou as validações do RAG.
Use a mesma .venv no Databricks; não há instalação automática ou downgrade de dependências.

### O que vamos avaliar
- Relevância dos documentos recuperados, com precision@3, @5 e @10 após rotulagem humana.
- Fidelidade de cada afirmação às fontes citadas e completude/clareza da resposta.
- Decisão de responder ou abster, usando referências confirmadas antes da rodada.
- Falhas de validação, erros técnicos, latência e consumo reportado pelo modelo.

Não haverá notas humanas inventadas, métricas de relevância sem rótulos nem inferência de fidelidade
apenas por um JSON válido. Não calculamos recall sem um conjunto de documentos relevantes de referência.

## Etapa 1 — Ambiente e caminhos no Databricks

### Objetivo
Avaliar o pipeline do Notebook 04 no mesmo Volume e modelo de chat escolhido.

### Justificativa
Cada rodada congela configuração, identificação declarada do modelo e hashes dos artefatos.
Resultados locais não devem ser retomados como execuções Databricks.

### Ação
Instale os pacotes como no 04, reinicie o Python e execute a preparação.

In [0]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version)

pacotes = [
    "sentence-transformers",
    "chromadb",
    "databricks-sdk",
    "openai",
    "python-dateutil",
    "botocore",
    "torch",
    "transformers",
    "pydantic",
]

for pacote in pacotes:
    try:
        print(f"{pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"{pacote}: não instalado")


In [0]:
from importlib.metadata import distributions
from pathlib import Path

restricoes = {}

for dist in distributions():
    nome = dist.metadata.get("Name")
    if nome and dist.version:
        restricoes[nome.lower().replace("_", "-")] = dist.version

arquivo = Path("/tmp/olist_rag_constraints.txt")
arquivo.write_text(
    "\n".join(
        f"{nome}=={versao}"
        for nome, versao in sorted(restricoes.items())
    ) + "\n",
    encoding="utf-8",
)

print(f"Restrições registradas: {len(restricoes)} pacotes.")

In [0]:
%pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu --constraint /tmp/olist_rag_constraints.txt

In [0]:
%pip install "sentence-transformers==5.1.0" "chromadb==1.0.20" "transformers==4.57.6" "openai>=1.0.0,<2.0.0" "torch==2.6.0" --constraint /tmp/olist_rag_constraints.txt

In [0]:
# Ativar as bibliotecas instaladas. Reexecute desde a célula seguinte após o reinício.
dbutils.library.restartPython()


In [0]:
from importlib.metadata import version

for pacote in [
    "packaging", "six", "python-dateutil", "databricks-sdk",
    "torch", "transformers", "sentence-transformers", "chromadb", "openai"
]:
    print(f"{pacote}: {version(pacote)}")

In [0]:
%pip check

In [0]:
# Objetivo:
# Conferir as dependências da avaliação.

# Justificativa:
# Reutilizar a ambiente Databricks do Notebook 04 sem alterar o banco ou seus pacotes.

# Ação:
# Mostrar o interpretador e as versões disponíveis.

import torch
import chromadb
import openai

from sentence_transformers import SentenceTransformer
from databricks.sdk import WorkspaceClient

print("Imports executados com sucesso.")
print("PyTorch:", torch.__version__)
print(
    "Cliente OpenAI do Databricks disponível:",
    hasattr(WorkspaceClient().serving_endpoints, "get_open_ai_client"),
)

In [0]:
# Objetivo:
# Definir os diretórios de entrada e avaliação.

# Justificativa:
# Separar dados preparados, código e evidências da avaliação sem alterar os notebooks anteriores.

# Ação:
# Localizar a raiz e criar somente as pastas de avaliação.


from pathlib import Path
import json
import os
import uuid
import importlib.util
import importlib.metadata
import inspect
import pandas as pd
import numpy as np
from IPython.display import display

import tempfile
import shutil
BASE_DIR = Path("/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4")
ARQUIVO_DOCUMENTOS = BASE_DIR / "data/processed/olist_reviews_documentos_rag.jsonl"
ARQUIVO_EMBEDDINGS = BASE_DIR / "data/processed/olist_reviews_embeddings.npy"
ARQUIVO_BUSCA = BASE_DIR / "src/busca_vetorial_databricks.py"
ARQUIVO_PIPELINE = BASE_DIR / "src/rag_pipeline_databricks.py"
PASTA_AVALIACAO = BASE_DIR / "reports/notebook05_databricks"
PASTA_PLANOS = PASTA_AVALIACAO / "planos"
PASTA_PLANOS.mkdir(parents=True, exist_ok=True)
for arquivo in [ARQUIVO_DOCUMENTOS, ARQUIVO_EMBEDDINGS, ARQUIVO_BUSCA, ARQUIVO_PIPELINE]:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Arquivo necessário não encontrado: {arquivo}. Confira os notebooks 03 e 04 Databricks.")
print("Projeto:", BASE_DIR)
print("Avaliação:", PASTA_AVALIACAO)

In [0]:
# Objetivo:
# Configurar a rodada de avaliação no Databricks.

# Justificativa:
# Controlar custo de tempo e permitir retomar um lote sem repetir inferências.

# Ação:
# Preencher o modelo e habilitar a avaliação somente quando desejar.


LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
LLM_MODELO_VERSAO = "Meta Llama 3.3 70B Instruct; endpoint gerenciado Databrick"
EXECUTAR_AVALIACAO = True #False
EXECUCAO_ID = "20261107T203256Z_3de570ec"  # Nova rodada Databricks; não usar o ID local.
MAX_PERGUNTAS_POR_EXECUCAO = 3
INTERROMPER_EM_ERRO_TECNICO = False
QUANTIDADE_CANDIDATOS = 12
MAX_DOCUMENTOS_CONTEXTO = 10
MAX_CARACTERES_CONTEXTO = 12000
MAX_TOKENS_RESPOSTA = 1400
LLM_TIMEOUT_S = 300
KS_AVALIACAO = (3, 5, 10)
ARQUIVO_PLANO = PASTA_PLANOS / "perguntas_avaliacao_v1.csv"
print("Consultas reais habilitadas:", EXECUTAR_AVALIACAO)

## Etapa 2 — Conjunto de perguntas e critérios de referência

O plano inicial contém **24 perguntas**: 16 qualitativas sobre avaliações, cinco fora do escopo e três
que exigem estatísticas que o RAG textual não fornece.

Os estados esperados são propostas de referência, não resultados do modelo:
- Nas 16 perguntas qualitativas, o status inicial é revisar: confirme se a base contém evidências adequadas.
- Nas demais, propomos fora_escopo ou evidencias_insuficientes, mas a referência também precisa ser confirmada.

Antes de consultar o LLM, abra o CSV, revise pergunta e critério de aceite, defina status_esperado como
respondido, evidencias_insuficientes ou fora_escopo e marque referencia_confirmada=1 somente após revisão.
Use avaliações da base para orientar a referência; não use a própria resposta do modelo como verdade.
Opcionalmente, faça revisão por duas pessoas e resolva divergências antes da rodada.

É permitido explorar o pipeline com referências pendentes, porém as métricas de status considerarão
somente as confirmadas no plano que foi congelado no início daquela rodada. Se mudar o plano depois,
crie outra execução: não se altera retroativamente a referência para favorecer o resultado observado.


In [0]:
# Objetivo:
# Preparar um plano inicial de avaliação.

# Justificativa:
# Cobrir temas distintos e casos de abstenção com critérios explícitos antes das inferências.

# Ação:
# Criar o CSV somente se ele ainda não existir, preservando suas revisões.

perguntas_propostas = [{'pergunta_id': 'Q01', 'tema': 'entrega_atraso', 'pergunta': 'O que os clientes relatam sobre atraso na entrega?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Citar relatos explícitos de atraso, sem confundir entrega rápida ou produto errado.'}, {'pergunta_id': 'Q02', 'tema': 'entrega_atraso', 'pergunta': 'Quais consequências do atraso são descritas nos comentários?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Somente consequências expressas; não inferir perdas ou danos não mencionados.'}, {'pergunta_id': 'Q03', 'tema': 'entrega_nao_recebida', 'pergunta': 'Há relatos recuperados de pedidos que ainda não chegaram?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Distinguir não recebimento de simples atraso já resolvido.'}, {'pergunta_id': 'Q04', 'tema': 'entrega_prazo', 'pergunta': 'Como os clientes descrevem o cumprimento do prazo de entrega?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Apresentar o que os relatos sustentam e reconhecer possível diversidade de experiências.'}, {'pergunta_id': 'Q05', 'tema': 'produto_defeito', 'pergunta': 'Quais relatos recuperados descrevem produtos quebrados ou com defeito?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Exigir menção a defeito ou quebra, não apenas atraso.'}, {'pergunta_id': 'Q06', 'tema': 'produto_qualidade', 'pergunta': 'O que os clientes relatam sobre a qualidade dos produtos recebidos?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Limitar conclusões a aspectos presentes nas fontes.'}, {'pergunta_id': 'Q07', 'tema': 'produto_estado', 'pergunta': 'Há comentários sobre produtos danificados ao chegar?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Citar descrições do dano e não atribuir causa logística sem evidência.'}, {'pergunta_id': 'Q08', 'tema': 'produto_funcionamento', 'pergunta': 'Quais experiências relatam problemas no funcionamento do produto?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Distinguir falha de funcionamento de item não recebido.'}, {'pergunta_id': 'Q09', 'tema': 'produto_errado', 'pergunta': 'Quais relatos mencionam produto diferente do que foi pedido?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Usar evidência de divergência do item; não tratar como defeito automaticamente.'}, {'pergunta_id': 'Q10', 'tema': 'quantidade_errada', 'pergunta': 'O que os clientes relatam sobre receber menos itens do que compraram?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Usar menção explícita à quantidade ou a item faltante.'}, {'pergunta_id': 'Q11', 'tema': 'embalagem', 'pergunta': 'O que os comentários recuperados dizem sobre a embalagem?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Distinguir condição da embalagem e condição do produto.'}, {'pergunta_id': 'Q12', 'tema': 'atendimento', 'pergunta': 'Quais experiências de atendimento aparecem nos relatos recuperados?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Não inventar respostas de SAC, protocolos ou resolução.'}, {'pergunta_id': 'Q13', 'tema': 'elogio_entrega', 'pergunta': 'Quais relatos elogiam a rapidez da entrega?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Exigir elogio à rapidez, não apenas avaliação positiva genérica.'}, {'pergunta_id': 'Q14', 'tema': 'elogio_produto', 'pergunta': 'Quais características dos produtos são elogiadas nos comentários?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Relatar somente características citadas, sem inferir satisfação pela nota sozinha.'}, {'pergunta_id': 'Q15', 'tema': 'elogio_experiencia', 'pergunta': 'O que os clientes elogiam na experiência de compra nos relatos recuperados?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Qualificar a síntese como observação nos relatos, não ranking de toda a base.'}, {'pergunta_id': 'Q16', 'tema': 'evidencias', 'pergunta': 'Quais avaliações sustentam relatos de problemas com a entrega?', 'status_esperado': 'revisar', 'referencia_confirmada': '0', 'criterio_aceite': 'Apresentar fontes rastreáveis com texto coerente ao tema.'}, {'pergunta_id': 'Q17', 'tema': 'fora_escopo', 'pergunta': 'Qual é a previsão do tempo para amanhã em Guarulhos?', 'status_esperado': 'fora_escopo', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: não há previsão meteorológica nas avaliações.'}, {'pergunta_id': 'Q18', 'tema': 'fora_escopo', 'pergunta': 'Qual é a cotação atual do dólar?', 'status_esperado': 'fora_escopo', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: não consultar conhecimento externo sobre câmbio.'}, {'pergunta_id': 'Q19', 'tema': 'fora_escopo', 'pergunta': 'Quem vencerá a próxima Copa do Mundo?', 'status_esperado': 'fora_escopo', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: tema externo e previsão não sustentada.'}, {'pergunta_id': 'Q20', 'tema': 'fora_escopo', 'pergunta': 'Qual remédio devo tomar para dor de cabeça?', 'status_esperado': 'fora_escopo', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: não fornecer orientação médica baseada em avaliações comerciais.'}, {'pergunta_id': 'Q21', 'tema': 'estatistica_global', 'pergunta': 'Qual o percentual de reclamações de atraso em toda a base?', 'status_esperado': 'evidencias_insuficientes', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: top-k não mede prevalência no corpus inteiro.'}, {'pergunta_id': 'Q22', 'tema': 'estatistica_global', 'pergunta': 'Qual foi a evolução mensal da taxa de defeitos em toda a base?', 'status_esperado': 'evidencias_insuficientes', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: exige classificação e agregação temporal da base completa.'}, {'pergunta_id': 'Q23', 'tema': 'informacao_ausente', 'pergunta': 'Qual é o faturamento atual da Olist?', 'status_esperado': 'fora_escopo', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: fato corporativo atual não fornecido pelo corpus de avaliações.'}, {'pergunta_id': 'Q24', 'tema': 'estatistica_global', 'pergunta': 'Qual transportadora teve a maior taxa de atrasos em toda a base?', 'status_esperado': 'evidencias_insuficientes', 'referencia_confirmada': '0', 'criterio_aceite': 'Abster: não há cálculo agregado por transportadora no pipeline.'}]
if not ARQUIVO_PLANO.exists():
    pd.DataFrame(perguntas_propostas).to_csv(ARQUIVO_PLANO, index=False, encoding="utf-8-sig")
print("Revise o plano antes de habilitar consultas reais:", ARQUIVO_PLANO)
display(pd.read_csv(ARQUIVO_PLANO, dtype=str, keep_default_na=False, encoding="utf-8-sig"))


In [0]:
# Objetivo:
# Definir funções de integridade e persistência.

# Justificativa:
# Vincular resultados à configuração correta e preservar arquivos mesmo em execuções parciais.

# Ação:
# Calcular hashes e gravar JSON por substituição local.

"""Avaliação local do RAG: persistência, revisão humana e métricas com denominadores explícitos."""
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

STATUS_VALIDOS = {"respondido", "evidencias_insuficientes", "fora_escopo", "validacao_falhou"}
STATUS_REFERENCIA = {"respondido", "evidencias_insuficientes", "fora_escopo"}



def hash_json(objeto):
    return hashlib.sha256(json.dumps(objeto, ensure_ascii=False, sort_keys=True, allow_nan=False).encode()).hexdigest()


def hash_arquivo(arquivo):
    h = hashlib.sha256()
    with Path(arquivo).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def gravar_json(arquivo, objeto):
    arquivo = Path(arquivo)
    temporario = arquivo.with_name(arquivo.name + ".tmp")
    temporario.write_text(json.dumps(objeto, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporario, arquivo)


In [0]:
# Objetivo:
# Validar o plano revisado.

# Justificativa:
# Impedir IDs duplicados, campos inválidos e referências confirmadas sem status definido.

# Ação:
# Ler o CSV e aplicar seu contrato.

def validar_perguntas(df):
    exigidas = ["pergunta_id", "tema", "pergunta", "status_esperado", "referencia_confirmada", "criterio_aceite"]
    if list(df.columns) != exigidas or df.empty:
        raise ValueError("Arquivo de perguntas vazio ou com colunas diferentes do modelo.")
    if df.pergunta_id.duplicated().any() or not df.pergunta_id.str.fullmatch(r"Q\d{2}").all():
        raise ValueError("IDs de pergunta inválidos ou repetidos.")
    if df.pergunta.str.strip().eq("").any() or df.criterio_aceite.str.strip().eq("").any():
        raise ValueError("Perguntas e critérios de aceite não podem estar vazios.")
    if not df.referencia_confirmada.isin(["0", "1"]).all():
        raise ValueError("referencia_confirmada precisa ser 0 ou 1.")
    if not df.status_esperado.isin(STATUS_REFERENCIA | {"revisar"}).all():
        raise ValueError("status_esperado inválido.")
    if ((df.referencia_confirmada == "1") & (df.status_esperado == "revisar")).any():
        raise ValueError("Uma referência confirmada precisa ter um status esperado definido.")


plano = pd.read_csv(ARQUIVO_PLANO, dtype=str, keep_default_na=False, encoding="utf-8-sig")
validar_perguntas(plano)
perguntas = plano.to_dict("records")
print("Perguntas:", len(perguntas))
print("Referências confirmadas:", int(plano.referencia_confirmada.eq("1").sum()))
print("Não altere a referência depois de ver a resposta para melhorar a nota da mesma rodada.")


## Etapa 3 — Reutilização do pipeline e identificação da rodada

Não duplicaremos as funções do RAG. O módulo local do Notebook 04 contém responder_rag e criar_consulta_ollama;
a recuperação continua vindo de src/busca_vetorial.py. Uma mudança nesses arquivos muda a identidade da avaliação.

A busca e o LLM só serão inicializados quando EXECUTAR_AVALIACAO=True. Assim, é possível analisar os
registros e preencher revisões em outra sessão sem manter endpoint Databricks ou o modelo de embeddings carregados.
Não serão feitas chamadas em lote apenas por importar o módulo.


In [0]:
import sys

In [0]:
# Objetivo:
# Importar o pipeline do Notebook 04 sem executar consultas.

# Justificativa:
# Avaliar exatamente o código utilizado no projeto.

# Ação:
# Carregar o módulo e conferir as funções públicas esperadas.


especificacao = importlib.util.spec_from_file_location("olist_pipeline_avaliacao05", ARQUIVO_PIPELINE)
if especificacao is None or especificacao.loader is None:
    raise ImportError("Não foi possível carregar rag_pipeline_databricks.py.")
modulo_rag = importlib.util.module_from_spec(especificacao)
sys.modules[especificacao.name] = modulo_rag
especificacao.loader.exec_module(modulo_rag)
for nome in ["responder_rag", "criar_consulta_databricks"]:
    if not callable(getattr(modulo_rag, nome, None)):
        raise AttributeError(f"O módulo do 04 não contém {nome}. Execute a última célula do 04 Databricks.")
parametros_resposta = inspect.signature(modulo_rag.responder_rag).parameters
if not {"pergunta", "recuperar", "consultar_llm"}.issubset(parametros_resposta):
    raise TypeError("A interface do pipeline mudou. Revise o adaptador antes de avaliar.")
print("Pipeline carregado. Versão:", getattr(modulo_rag, "VERSAO_PIPELINE", "não informada"))

In [0]:
# Objetivo:
# Definir o manifesto de cada rodada.

# Justificativa:
# Não misturar perguntas, código, base ou configuração de modelos distintos.

# Ação:
# Congelar uma assinatura e rejeitar alterações ao retomar.

def abrir_execucao(pasta, configuracao, perguntas):
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    dados = {"schema_version": 1, "configuracao": configuracao, "perguntas": perguntas}
    assinatura = hash_json(dados)
    arquivo = pasta / "manifesto_execucao.json"
    if arquivo.exists():
        existente = json.loads(arquivo.read_text(encoding="utf-8"))
        conteudo_existente = {k: existente[k] for k in ["schema_version", "configuracao", "perguntas"]}
        if hash_json(conteudo_existente) != existente.get("assinatura"):
            raise ValueError("O manifesto foi alterado depois da criação.")
        if existente.get("assinatura") != assinatura:
            raise ValueError("Configuração, dados, modelo ou perguntas mudaram. Use um novo EXECUCAO_ID; não misture execuções.")
        return existente
    dados.update(assinatura=assinatura, criado_em_utc=datetime.now(timezone.utc).isoformat())
    gravar_json(arquivo, dados)
    return dados


In [0]:
# Objetivo: reconstruir somente o cache local do driver, a partir dos artefatos do Volume.
# Justificativa: notebooks/sessões podem não compartilhar o mesmo disco temporário.
# Ação: validar a base, reconstituir a coleção e reutilizar o módulo do Notebook 03.
def inicializar_recuperacao_databricks():
    # Objetivo:
    # Recarregar e conferir os documentos e vetores.

    # Justificativa:
    # Evitar recuperar um vetor associado a outro documento.

    # Ação:
    # Validar integridade estrutural antes de criar o índice.


    df_documentos_rag = pd.read_json(
        ARQUIVO_DOCUMENTOS, lines=True, convert_dates=False,
        dtype={"documento_id": "str", "review_id": "str", "order_id": "str",
               "review_score": "int64", "review_creation_date": "str", "texto_documento": "str"}
    )
    colunas = ["documento_id", "review_id", "order_id", "review_score", "review_creation_date", "texto_documento"]
    if df_documentos_rag.empty or not set(colunas).issubset(df_documentos_rag):
        raise ValueError("Base vazia ou sem as colunas obrigatórias.")
    if df_documentos_rag[colunas].isna().any().any():
        raise ValueError("Há campos obrigatórios ausentes.")
    if df_documentos_rag.documento_id.duplicated().any() or df_documentos_rag.texto_documento.str.strip().eq("").any():
        raise ValueError("Há documentos duplicados por ID ou textos vazios.")
    embeddings_documentos = np.load(ARQUIVO_EMBEDDINGS, allow_pickle=False)
    if embeddings_documentos.shape != (len(df_documentos_rag), 384) or embeddings_documentos.dtype != np.float32:
        raise ValueError("O NPY precisa ser float32, com uma linha por documento e 384 colunas.")
    if not np.isfinite(embeddings_documentos).all() or (np.linalg.norm(embeddings_documentos, axis=1) == 0).any():
        raise ValueError("A matriz contém valores não finitos ou vetores nulos.")
    identidade = hashlib.sha256()
    for arquivo in [ARQUIVO_DOCUMENTOS, ARQUIVO_EMBEDDINGS]:
        with arquivo.open("rb") as stream:
            for bloco in iter(lambda: stream.read(1024 * 1024), b""):
                identidade.update(bloco)
    HASH_BASE = identidade.hexdigest()
    DIRETORIO_BANCO_VETORIAL = Path(tempfile.gettempdir()) / "olist_fase4" / HASH_BASE / "vector_store"
    os.environ["OLIST_CHROMA_DIR"] = str(DIRETORIO_BANCO_VETORIAL)
    print("Documentos:", len(df_documentos_rag))
    print("Embeddings:", embeddings_documentos.shape)
    print("Identidade dos artefatos:", HASH_BASE)
    print("Cache Chroma:", DIRETORIO_BANCO_VETORIAL)


    # Objetivo:
    # Recuperar a coleção Chroma nesta sessão.

    # Justificativa:
    # Permitir consultar o RAG mesmo após o encerramento da sessão do Notebook 03.

    # Ação:
    # Inserir os vetores existentes em lotes somente quando a coleção estiver incompleta.


    import chromadb
    cliente_chroma = chromadb.PersistentClient(path=str(DIRETORIO_BANCO_VETORIAL))
    colecao_reviews = cliente_chroma.get_or_create_collection(
        name="olist_reviews_rag", metadata={"hnsw:space": "cosine"}
    )
    if colecao_reviews.count() > len(df_documentos_rag):
        raise ValueError("Coleção tem registros extras. Confira o cache associado ao hash da base.")
    if colecao_reviews.count() != len(df_documentos_rag):
        lote = min(512, cliente_chroma.get_max_batch_size())
        for inicio in range(0, len(df_documentos_rag), lote):
            fim = inicio + lote
            parte = df_documentos_rag.iloc[inicio:fim]
            metadados = [{"review_id": str(r.review_id), "order_id": str(r.order_id),
                         "review_score": int(r.review_score), "review_creation_date": str(r.review_creation_date)}
                        for r in parte.itertuples(index=False)]
            colecao_reviews.upsert(ids=parte.documento_id.tolist(), documents=parte.texto_documento.tolist(),
                                   metadatas=metadados, embeddings=embeddings_documentos[inicio:fim].tolist())
        print("Coleção reconstruída a partir dos artefatos persistidos.")
    else:
        print("Coleção completa encontrada no cache desta sessão.")
    assert colecao_reviews.count() == len(df_documentos_rag)
    for pos in [0, len(df_documentos_rag) // 2, len(df_documentos_rag) - 1]:
        linha = df_documentos_rag.iloc[pos]
        registro = colecao_reviews.get(ids=[linha.documento_id], include=["documents", "metadatas", "embeddings"])
        assert registro["documents"][0] == linha.texto_documento
        assert registro["metadatas"][0]["review_id"] == linha.review_id
        np.testing.assert_allclose(registro["embeddings"][0], embeddings_documentos[pos], rtol=1e-5, atol=1e-6)
    print("Contagem e integridade das amostras aprovadas.")


    # Objetivo:
    # Reutilizar o módulo de busca gerado no Notebook 03.

    # Justificativa:
    # Não substituir a recuperação já validada por outra implementação.

    # Ação:
    # Importar uma cópia local do módulo persistido no Volume.


    arquivo_local = Path(tempfile.mkdtemp(prefix="olist_busca04_")) / ARQUIVO_BUSCA.name
    shutil.copyfile(ARQUIVO_BUSCA, arquivo_local)
    especificacao = importlib.util.spec_from_file_location("olist_busca_nb04", arquivo_local)
    if especificacao is None or especificacao.loader is None:
        raise ImportError("Não foi possível criar o carregador do módulo de busca.")
    modulo_busca = importlib.util.module_from_spec(especificacao)
    sys.modules[especificacao.name] = modulo_busca
    especificacao.loader.exec_module(modulo_busca)
    modelo_embeddings, colecao_modulo = modulo_busca.inicializar_busca_vetorial()
    assert modulo_busca.DIRETORIO_BANCO_VETORIAL.resolve() == DIRETORIO_BANCO_VETORIAL.resolve()
    assert colecao_modulo.count() == len(df_documentos_rag)
    # Diagnóstico amostral de compatibilidade do modelo com os vetores persistidos.
    # Não equivale à verificação integral da proveniência de todos os vetores.
    indices = [0, len(df_documentos_rag) // 2, len(df_documentos_rag) - 1]
    vetores_teste = modelo_embeddings.encode(df_documentos_rag.iloc[indices].texto_documento.tolist(), convert_to_numpy=True)
    if not np.allclose(vetores_teste, embeddings_documentos[indices], rtol=1e-4, atol=1e-5):
        raise ValueError("O modelo carregado não reproduziu as amostras do NPY. Confira a versão do modelo ou reconstrua no 03.")

    def recuperar_avaliacoes(pergunta, quantidade=12):
        return modulo_busca.buscar_avaliacoes(
            pergunta=pergunta, modelo=modelo_embeddings, colecao=colecao_modulo,
            quantidade=min(quantidade, colecao_modulo.count())
        )
    print("Busca do Notebook 03 disponível nesta sessão.")

    return recuperar_avaliacoes, modulo_busca.NOME_MODELO

## Etapa 4 — Execução controlada e retomada

Na primeira rodada, deixe MAX_PERGUNTAS_POR_EXECUCAO=3. Após conferir os resultados e o tempo,
execute novamente para avançar nas perguntas restantes. Para outra sessão, cole em EXECUCAO_ID
o identificador que foi impresso na criação da rodada.

Resultados existentes não são recalculados. Erros técnicos são preservados e o lote para por padrão.
Após corrigir um erro ou ajustar o modelo/prompt, crie outra execução para comparar; não apague tentativas
a fim de selecionar apenas respostas favoráveis. O resultado validacao_falhou é uma falha observada do
pipeline, não equivale a uma abstenção correta.

Alterações no plano, versões dos pacotes, hashes dos dados/código ou digest do modelo impedem retomar
uma rodada com configuração diferente. Nenhum arquivo do 03 ou do 04 será substituído.


In [0]:
# Objetivo:
# Definir a execução em lotes.

# Justificativa:
# Evitar chamadas repetidas e registrar a tentativa antes de avançar.

# Ação:
# Recuperar uma vez por pergunta e fornecer os mesmos candidatos ao pipeline.

def executar_lote(pasta, manifesto, perguntas, recuperar, consultar, responder,
                   quantidade=12, max_documentos=10, max_caracteres=12000,
                   limite=3, interromper_erro=True):
    if limite < 1:
        raise ValueError("limite precisa ser positivo.")
    pasta = Path(pasta)
    destinos = pasta / "resultados"
    destinos.mkdir(exist_ok=True)
    feitos = []
    for questao in perguntas:
        arquivo = destinos / (questao["pergunta_id"] + ".json")
        if arquivo.exists():
            anterior = json.loads(arquivo.read_text(encoding="utf-8"))
            if anterior.get("assinatura_execucao") != manifesto["assinatura"]:
                raise ValueError("Resultado pertence a outra assinatura de execução.")
            continue
        inicio = time.perf_counter()
        registro = {"pergunta_id": questao["pergunta_id"], "pergunta": questao["pergunta"],
                    "assinatura_execucao": manifesto["assinatura"],
                    "criado_em_utc": datetime.now(timezone.utc).isoformat(),
                    "candidatos_brutos": [], "resultado": None, "erro_tecnico": None}
        try:
            brutos = recuperar(questao["pergunta"], quantidade)
            registro["candidatos_brutos"] = brutos
            resultado = responder(questao["pergunta"], lambda p, k: brutos, consultar,
                                  quantidade, max_documentos, max_caracteres)
            if resultado.get("status") not in STATUS_VALIDOS:
                raise ValueError("Pipeline retornou um status desconhecido.")
            registro["resultado"] = resultado
        except Exception as exc:
            # Não registrar a mensagem completa: respostas HTTP podem incluir detalhes do ambiente.
            registro["erro_tecnico"] = type(exc).__name__
        registro["latencia_externa_s"] = time.perf_counter() - inicio
        gravar_json(arquivo, registro)
        feitos.append(questao["pergunta_id"])
        print(questao["pergunta_id"], "erro_tecnico" if registro["erro_tecnico"] else registro["resultado"]["status"])
        if registro["erro_tecnico"] and interromper_erro:
            print("Lote interrompido. Corrija a causa técnica; a tentativa foi preservada e não será repetida automaticamente.")
            break
        if len(feitos) >= limite:
            break
    return feitos


def carregar_resultados(pasta, manifesto):
    esperado = {k: manifesto[k] for k in ["schema_version", "configuracao", "perguntas"]}
    if hash_json(esperado) != manifesto.get("assinatura"):
        raise ValueError("O manifesto foi alterado depois da criação da rodada.")
    registros = []
    ids_permitidos = {q["pergunta_id"] for q in manifesto["perguntas"]}
    for arquivo in sorted((Path(pasta) / "resultados").glob("Q*.json")):
        registro = json.loads(arquivo.read_text(encoding="utf-8"))
        if (registro.get("assinatura_execucao") != manifesto["assinatura"]
            or registro.get("pergunta_id") not in ids_permitidos
            or arquivo.stem != registro["pergunta_id"]):
            raise ValueError(f"Resultado incompatível: {arquivo.name}")
        registros.append(registro)
    return registros


In [0]:
# Objetivo:
# Executar ou abrir a rodada selecionada.

# Justificativa:
# Separar a consulta real da análise posterior dos arquivos.

# Ação:
# Inicializar a busca e o endpoint Databricks apenas quando habilitado; congelar o manifesto e rodar um lote.


manifesto = None
PASTA_EXECUCAO = None
registros = []
if EXECUCAO_ID and (not EXECUCAO_ID.replace("-", "").replace("_", "").isalnum()):
    raise ValueError("EXECUCAO_ID deve conter apenas letras, números, hífen ou underscore.")
if EXECUTAR_AVALIACAO:
    recuperar_avaliacoes, nome_embeddings = inicializar_recuperacao_databricks()
    consulta_llm = modulo_rag.criar_consulta_databricks(
        LLM_ENDPOINT, MAX_TOKENS_RESPOSTA, LLM_TIMEOUT_S, LLM_MODELO_VERSAO)
    ambiente = {nome: importlib.metadata.version(nome) for nome in
        ["pandas", "numpy", "chromadb", "sentence-transformers", "databricks-sdk", "openai"]}
    configuracao = {
        **consulta_llm.metadados_modelo, "ambiente":ambiente,
        "modelo_embeddings":nome_embeddings,
        "hash_jsonl":hash_arquivo(ARQUIVO_DOCUMENTOS), "hash_npy":hash_arquivo(ARQUIVO_EMBEDDINGS),
        "hash_busca":hash_arquivo(ARQUIVO_BUSCA), "hash_pipeline":hash_arquivo(ARQUIVO_PIPELINE),
        "quantidade_candidatos":QUANTIDADE_CANDIDATOS, "max_documentos":MAX_DOCUMENTOS_CONTEXTO,
        "max_caracteres":MAX_CARACTERES_CONTEXTO, "max_tokens":MAX_TOKENS_RESPOSTA,
        "timeout_s":LLM_TIMEOUT_S, "temperatura":0.0,
    }
    if not EXECUCAO_ID:
        EXECUCAO_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    PASTA_EXECUCAO = PASTA_AVALIACAO / "execucoes" / EXECUCAO_ID
    manifesto = abrir_execucao(PASTA_EXECUCAO, configuracao, perguntas)
    print("EXECUCAO_ID =", repr(EXECUCAO_ID))
    print("Copie esse ID para a configuração ao retomar em outra sessão.")
    executar_lote(PASTA_EXECUCAO, manifesto, perguntas, recuperar_avaliacoes, consulta_llm,
        modulo_rag.responder_rag, QUANTIDADE_CANDIDATOS, MAX_DOCUMENTOS_CONTEXTO,
        MAX_CARACTERES_CONTEXTO, MAX_PERGUNTAS_POR_EXECUCAO, INTERROMPER_EM_ERRO_TECNICO)
elif EXECUCAO_ID:
    PASTA_EXECUCAO = PASTA_AVALIACAO / "execucoes" / EXECUCAO_ID
    arquivo_manifesto = PASTA_EXECUCAO / "manifesto_execucao.json"
    if not arquivo_manifesto.is_file():
        raise FileNotFoundError(f"Rodada não encontrada: {EXECUCAO_ID}")
    manifesto = json.loads(arquivo_manifesto.read_text(encoding="utf-8"))
    print("Rodada aberta para revisão sem inicializar modelos:", EXECUCAO_ID)
else:
    print("Nenhuma consulta real executada. Revise o plano e configure EXECUTAR_AVALIACAO=True quando desejar.")
if manifesto is not None:
    registros = carregar_resultados(PASTA_EXECUCAO, manifesto)
    print("Executadas:", len(registros), "/", len(manifesto["perguntas"]))

In [0]:
# Objetivo:
# Exibir um resumo técnico das perguntas executadas.

# Justificativa:
# Distinguir respostas, abstenções, falhas de validação e falhas de serviço.

# Ação:
# Mostrar status, chamadas e tempo sem atribuir notas automáticas de qualidade.


resumo_execucoes = []
for registro in registros:
    resultado = registro.get("resultado") or {}
    resumo_execucoes.append({"pergunta_id": registro["pergunta_id"], "pergunta": registro["pergunta"],
        "status": "erro_tecnico" if registro["erro_tecnico"] else resultado["status"],
        "motivo": registro["erro_tecnico"] or resultado.get("motivo"),
        "chamadas_reportadas": len(resultado.get("chamadas_llm", [])) if not registro["erro_tecnico"] else None,
        "latencia_s": registro["latencia_externa_s"]})
df_execucoes = pd.DataFrame(resumo_execucoes)
display(df_execucoes)


## Etapa 5 — Revisão humana da recuperação, das afirmações e das respostas

Serão criados três arquivos CSV dentro da rodada. Abra no Databricks ou em um editor que preserve UTF-8.
No Excel, importe os identificadores como texto para não alterar IDs e rankings. Use **0, 1 ou vazio**
nos campos de julgamento; vazio significa pendente, nunca zero.

| Arquivo | Campos editáveis | Critério |
| --- | --- | --- |
| revisao_recuperacao.csv | relevante, justificativa | 1 se o comentário ajuda diretamente a responder; 0 se não ajuda. |
| revisao_afirmacoes.csv | sustentada, justificativa | 1 se a afirmação inteira é sustentada pelas fontes citadas; 0 se houver extrapolação, contradição ou atribuição errada. |
| revisao_respostas.csv | completa, clara, justificativa | Para status respondido: 1 se cobre o que a pergunta e as evidências permitem e se está clara; 0 se não. |

Não altere texto de comentário, afirmação, pergunta, status, documento_id ou chaves nesses arquivos.
O notebook verifica a integridade desses campos e rejeita a revisão se eles forem modificados.
Novos lotes acrescentam somente as novas linhas, sem apagar os julgamentos já preenchidos.

Para julgar sustentação, leia a afirmação e a avaliação completa, não apenas o trecho destacado.
Não deduza defeito a partir de atraso ou frequência populacional a partir de poucos comentários.
Completude é relativa ao que as evidências permitem responder, não significa inventar informações ausentes.
Nas abstenções, deixe completa/clara vazias; a correção do status será comparada com o plano congelado.


In [0]:
# Objetivo:
# Preparar e preservar os arquivos de revisão humana.

# Justificativa:
# Permitir rotular a evidência sem perder trabalho ao executar novos lotes.

# Ação:
# Construir tabelas de referência e validar os campos que não podem ser editados.

def preparar_tabelas_revisao(registros):
    recuperacao, afirmacoes, respostas = [], [], []
    for r in registros:
        qid = r["pergunta_id"]
        for rank, item in enumerate(r["candidatos_brutos"], start=1):
            recuperacao.append({"pergunta_id": qid, "rank": str(rank), "pergunta": r["pergunta"],
                "documento_id": item["documento_id"], "texto": item["texto"], "distancia": str(item["distancia"]),
                "relevante": "", "justificativa": ""})
        resultado = r.get("resultado") or {}
        fontes = {e["fonte"]: e for e in resultado.get("fontes_utilizadas", [])}
        for indice, a in enumerate(resultado.get("afirmacoes", []), start=1):
            evidencias = [{"fonte": f, "documento_id": fontes[f]["documento_id"], "trecho": fontes[f]["trecho"],
                           "texto_avaliacao": fontes[f]["texto"]} for f in a["fontes"] if f in fontes]
            afirmacoes.append({"pergunta_id": qid, "afirmacao_id": str(indice), "pergunta": r["pergunta"],
                "afirmacao": a["texto"], "fontes_e_trechos": json.dumps(evidencias, ensure_ascii=False),
                "sustentada": "", "justificativa": ""})
        respostas.append({"pergunta_id": qid, "pergunta": r["pergunta"],
            "status_observado": "erro_tecnico" if r["erro_tecnico"] else resultado.get("status", "erro_tecnico"),
            "resposta": json.dumps(resultado.get("afirmacoes", []), ensure_ascii=False),
            "fontes_utilizadas": json.dumps(resultado.get("fontes_utilizadas", []), ensure_ascii=False),
            "completa": "", "clara": "", "justificativa": ""})
    return {
        "revisao_recuperacao.csv": (pd.DataFrame(recuperacao, columns=["pergunta_id", "rank", "pergunta", "documento_id", "texto", "distancia", "relevante", "justificativa"]), ["pergunta_id", "rank"], ["relevante", "justificativa"]),
        "revisao_afirmacoes.csv": (pd.DataFrame(afirmacoes, columns=["pergunta_id", "afirmacao_id", "pergunta", "afirmacao", "fontes_e_trechos", "sustentada", "justificativa"]), ["pergunta_id", "afirmacao_id"], ["sustentada", "justificativa"]),
        "revisao_respostas.csv": (pd.DataFrame(respostas, columns=["pergunta_id", "pergunta", "status_observado", "resposta", "fontes_utilizadas", "completa", "clara", "justificativa"]), ["pergunta_id"], ["completa", "clara", "justificativa"]),
    }


def atualizar_e_ler_revisao(arquivo, modelo, chaves, editaveis):
    """Acrescenta linhas de novos lotes sem perder julgamentos; rejeita alteração dos dados avaliados."""
    arquivo = Path(arquivo)
    if not arquivo.exists():
        modelo.to_csv(arquivo, index=False, encoding="utf-8-sig")
        return modelo.copy()
    revisao = pd.read_csv(arquivo, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if list(revisao.columns) != list(modelo.columns) or revisao.duplicated(chaves).any():
        raise ValueError(f"Colunas ou chaves alteradas em {arquivo.name}.")
    identidade = lambda df: list(map(tuple, df[chaves].astype(str).to_numpy()))
    esperadas = dict(zip(identidade(modelo), modelo.to_dict("records")))
    existentes = set()
    for linha in revisao.to_dict("records"):
        chave = tuple(linha[c] for c in chaves)
        if chave not in esperadas:
            raise ValueError(f"Linha desconhecida em {arquivo.name}.")
        existentes.add(chave)
        for campo in modelo.columns:
            if campo not in editaveis and str(linha[campo]).replace("\r\n", "\n") != str(esperadas[chave][campo]).replace("\r\n", "\n"):
                raise ValueError(f"Campo de evidência alterado em {arquivo.name}: {campo}.")
        for campo in editaveis:
            if campo != "justificativa" and linha[campo] not in {"", "0", "1"}:
                raise ValueError(f"Rótulo inválido em {arquivo.name}/{campo}; use vazio, 0 ou 1.")
    novas = [linha for chave, linha in esperadas.items() if chave not in existentes]
    if novas:
        revisao = pd.concat([revisao, pd.DataFrame(novas, columns=modelo.columns)], ignore_index=True)
        revisao.to_csv(arquivo, index=False, encoding="utf-8-sig")
    return revisao

In [0]:
# Objetivo:
# Criar ou reler as revisões da rodada.

# Justificativa:
# Usar apenas julgamentos explícitos e manter rastreabilidade por pergunta e documento.

# Ação:
# Acrescentar linhas novas e carregar os CSVs já preenchidos.


tabelas_revisao = preparar_tabelas_revisao(registros)
revisoes = {}
for nome, (modelo, chaves, editaveis) in tabelas_revisao.items():
    if PASTA_EXECUCAO is not None:
        revisoes[nome] = atualizar_e_ler_revisao(PASTA_EXECUCAO / nome, modelo, chaves, editaveis)
        print("Revisão:", PASTA_EXECUCAO / nome)
    else:
        revisoes[nome] = modelo
revisao_recuperacao = revisoes["revisao_recuperacao.csv"]
revisao_afirmacoes = revisoes["revisao_afirmacoes.csv"]
revisao_respostas = revisoes["revisao_respostas.csv"]
display(revisao_recuperacao.head())
print("Preencha os CSVs, salve-os e reexecute esta célula e as etapas de métricas/relatório.")


## Etapa 6 — Métricas e cobertura da avaliação

### Recuperação
Precision@k = documentos relevantes nos primeiros k resultados / k, com k fixo em 3, 5 e 10.
Se houver menos de k documentos retornados, as posições ausentes contam como zero. Se não houver
candidatos, a precisão é zero. Se algum documento retornado no prefixo k estiver sem rótulo,
a métrica daquela pergunta/k fica pendente. A média principal considera somente perguntas com referência confirmada como respondido, para não
misturar precisão de recuperação em casos respondíveis com perguntas fora de escopo. Também é apresentada
uma média diagnóstica de todas as perguntas revisadas, com esse denominador explicitamente separado.
Cada média informa quantas perguntas têm o prefixo revisado.
A recuperação avaliada é a ordem bruta de vizinhos do 03; o contexto deduplicado do RAG é registrado à parte.
Não há recall, MAP ou nDCG sem referência adequada para essas métricas.

### Respostas
A fidelidade micro é a proporção de afirmações sustentadas entre as **afirmações revisadas**.
Sempre mostramos o total revisado e o total de afirmações, para não ocultar uma revisão parcial.
Completude e clareza são proporções entre respostas respondidas e revisadas em cada critério.
A cobertura estrutural de citações verifica, para cada afirmação, se há referências e se os trechos
correspondem a documentos recuperados; ela não mede se a afirmação é sustentada semanticamente.
Não usar aprovação estrutural de citações como substituto de sustentação semântica.

### Status e desempenho
Acurácia de status usa apenas referências confirmadas no plano congelado, com denominador explícito.
A taxa de abstenção correta exige o status esperado exato; erro técnico e validacao_falhou não contam como acerto.
Latência p50 e p95 inclui todas as tentativas registradas, inclusive falhas, medidas ao redor da pergunta completa.
Tokens são somados somente quando reportados pelo pipeline. Chamadas com erro podem ter consumo desconhecido.
Não estimamos preço sem uma tabela de cobrança e não comparamos latência local com Databricks como se fosse o mesmo hardware.


In [0]:
# Objetivo:
# Calcular métricas somente onde há dados suficientes.

# Justificativa:
# Evitar notas inventadas e apresentar denominadores e pendências da avaliação.

# Ação:
# Consolidar status, precisão, revisão humana e desempenho.

def auditar_citacoes(registros):
    linhas = []
    normalizar = lambda x: " ".join(str(x).split())
    for registro in registros:
        resultado = registro.get("resultado") or {}
        fontes = {e["fonte"]: e for e in resultado.get("fontes_utilizadas", [])}
        documentos = {d["documento_id"]: d for d in registro["candidatos_brutos"]}
        for indice, afirmacao in enumerate(resultado.get("afirmacoes", []), start=1):
            validas = []
            for ref in afirmacao.get("fontes", []):
                fonte = fontes.get(ref, {})
                documento = documentos.get(fonte.get("documento_id"), {})
                trecho = normalizar(fonte.get("trecho", ""))
                validas.append(bool(documento) and bool(trecho)
                    and trecho in normalizar(documento.get("texto", ""))
                    and normalizar(fonte.get("texto", "")) == normalizar(documento.get("texto", "")))
            linhas.append({"pergunta_id": registro["pergunta_id"], "afirmacao_id": str(indice),
                           "citacao_estrutural_valida": bool(validas) and all(validas)})
    return pd.DataFrame(linhas, columns=["pergunta_id", "afirmacao_id", "citacao_estrutural_valida"])


def calcular_metricas(registros, perguntas, recuperacao, afirmacoes, respostas, ks=(3, 5, 10)):
    resumo = {"perguntas_planejadas": len(perguntas), "perguntas_executadas": len(registros)}
    resumo["cobertura_execucao"] = len(registros) / len(perguntas) if perguntas else None
    linhas_status, tempos, tokens_conhecidos = [], [], 0
    chamadas_com_uso, chamadas_sem_uso, erros = 0, 0, 0
    for r in registros:
        resultado = r.get("resultado") or {}
        status = "erro_tecnico" if r["erro_tecnico"] else resultado.get("status", "erro_tecnico")
        linhas_status.append({"pergunta_id": r["pergunta_id"], "status": status})
        erros += int(status == "erro_tecnico")
        tempos.append(r["latencia_externa_s"])
        for c in resultado.get("chamadas_llm", []):
            uso = c.get("uso") or {}
            total = uso.get("total_tokens")
            if isinstance(total, int) and not isinstance(total, bool) and total >= 0:
                tokens_conhecidos += total
                chamadas_com_uso += 1
            else:
                chamadas_sem_uso += 1
    resumo.update(distribuicao_status=pd.Series([x["status"] for x in linhas_status], dtype=str).value_counts().to_dict(),
                  erros_tecnicos=erros, p50_latencia_s=float(np.percentile(tempos, 50)) if tempos else None,
                  p95_latencia_s=float(np.percentile(tempos, 95)) if tempos else None,
                  tokens_total_reportado=tokens_conhecidos, chamadas_com_uso_reportado=chamadas_com_uso,
                  chamadas_sem_uso_reportado=chamadas_sem_uso,
                  observacao_consumo="Tentativas com erro podem ter consumo não registrado; não foi estimado preço.")
    metricas_p = []
    for qid in [r["pergunta_id"] for r in registros if not r["erro_tecnico"]]:
        parte = recuperacao.loc[recuperacao.pergunta_id == qid].copy()
        parte["ordem"] = pd.to_numeric(parte["rank"])
        for k in ks:
            top = parte.loc[parte.ordem <= k].sort_values("ordem")
            completo = top.relevante.isin(["0", "1"]).all()
            metricas_p.append({"pergunta_id": qid, "k": k, "documentos_retornados_ate_k": len(top),
                               "rotulados": int(top.relevante.isin(["0", "1"]).sum()),
                               "precision_at_k": float(top.relevante.eq("1").sum() / k) if completo else None})
    precisao = pd.DataFrame(metricas_p, columns=["pergunta_id", "k", "documentos_retornados_ate_k", "rotulados", "precision_at_k"])
    elegiveis = {q["pergunta_id"] for q in perguntas
                 if q["referencia_confirmada"] == "1" and q["status_esperado"] == "respondido"}
    medias = lambda tabela: {str(k): {
        "media": float(g.precision_at_k.mean()) if g.precision_at_k.notna().any() else None,
        "perguntas_com_topk_rotulado": int(g.precision_at_k.notna().sum())}
        for k, g in tabela.groupby("k")}
    resumo["precision_media_por_k_respondiveis_confirmadas"] = medias(precisao.loc[precisao.pergunta_id.isin(elegiveis)])
    resumo["precision_diagnostica_por_k_todas_perguntas"] = medias(precisao)
    resumo["perguntas_confirmadas_respondiveis_no_plano"] = len(elegiveis)
    auditoria = auditar_citacoes(registros)
    resumo["afirmacoes_com_citacao_estrutural_valida"] = int(auditoria.citacao_estrutural_valida.sum())
    resumo["cobertura_citacao_estrutural"] = float(auditoria.citacao_estrutural_valida.mean()) if len(auditoria) else None
    rotulos_afirmacoes = afirmacoes.sustentada.isin(["0", "1"])
    n_afirmacoes = int(rotulos_afirmacoes.sum())
    resumo["afirmacoes_totais"] = len(afirmacoes)
    resumo["afirmacoes_revisadas"] = n_afirmacoes
    resumo["fidelidade_nas_afirmacoes_revisadas"] = float(afirmacoes.loc[rotulos_afirmacoes, "sustentada"].eq("1").mean()) if n_afirmacoes else None
    resumo["afirmacoes_sem_suporte_revisadas"] = int(afirmacoes.loc[rotulos_afirmacoes, "sustentada"].eq("0").sum())
    for campo in ["completa", "clara"]:
        elegiveis = respostas.status_observado.eq("respondido")
        revisadas = elegiveis & respostas[campo].isin(["0", "1"])
        resumo[campo + "_taxa_nas_respostas_revisadas"] = float(respostas.loc[revisadas, campo].eq("1").mean()) if revisadas.any() else None
        resumo[campo + "_respostas_revisadas"] = int(revisadas.sum())
    # Usar somente referências confirmadas ANTES da rodada, guardadas no manifesto.
    ref = {q["pergunta_id"]: q for q in perguntas if q["referencia_confirmada"] == "1"}
    comparacoes = []
    for linha in linhas_status:
        if linha["pergunta_id"] in ref:
            esperado = ref[linha["pergunta_id"]]["status_esperado"]
            comparacoes.append({**linha, "status_esperado": esperado, "acertou_status": linha["status"] == esperado})
    resumo["referencias_confirmadas_no_plano"] = len(ref)
    resumo["perguntas_executadas_com_referencia_confirmada"] = len(comparacoes)
    resumo["acuracia_status_nas_referencias_executadas"] = float(np.mean([x["acertou_status"] for x in comparacoes])) if comparacoes else None
    # Acerto na decisão responder/abster: falha de validação ou erro não conta como abstenção correta.
    abster = [x for x in comparacoes if x["status_esperado"] in {"fora_escopo", "evidencias_insuficientes"}]
    resumo["casos_abstencao_esperada_executados"] = len(abster)
    resumo["abstencao_correta_status_exato"] = float(np.mean([x["acertou_status"] for x in abster])) if abster else None
    resumo["pendencias"] = {
        "perguntas_nao_executadas": len(perguntas) - len(registros),
        "referencias_nao_confirmadas_no_plano": len(perguntas) - len(ref),
        "documentos_sem_rotulo": int((~recuperacao.relevante.isin(["0", "1"])).sum()),
        "afirmacoes_sem_revisao": len(afirmacoes) - n_afirmacoes,
        "respostas_respondidas_sem_completude": int((respostas.status_observado.eq("respondido") & ~respostas.completa.isin(["0", "1"])).sum()),
        "respostas_respondidas_sem_clareza": int((respostas.status_observado.eq("respondido") & ~respostas.clara.isin(["0", "1"])).sum()),
    }
    resumo["estado_avaliacao"] = "com_pendencias" if any(resumo["pendencias"].values()) or erros else "revisao_preenchida"
    return resumo, precisao, pd.DataFrame(comparacoes)


In [0]:
# Objetivo:
# Exibir os resultados e suas pendências.

# Justificativa:
# Distinguir uma execução parcial de uma avaliação com todas as revisões preenchidas.

# Ação:
# Calcular contra as referências do manifesto, e não contra um plano editado depois.


perguntas_da_rodada = manifesto["perguntas"] if manifesto else perguntas
metricas, precisao_por_pergunta, comparacao_status = calcular_metricas(
    registros, perguntas_da_rodada, revisao_recuperacao, revisao_afirmacoes,
    revisao_respostas, KS_AVALIACAO
)
print(json.dumps(metricas, ensure_ascii=False, indent=2))
display(precisao_por_pergunta)
display(comparacao_status)
print("revisao_preenchida significa preenchimento completo; não significa qualidade aprovada nem ausência de falhas.")


## Etapa 7 — Testes do código de avaliação sem modelos

As fixtures seguintes são sintéticas. Testam denominadores, ausência de rótulos, preservação da revisão
e retomada de lote. Não são métricas do corpus Olist nem resultados do qwen2.5:3b.
endpoint Databricks, Chroma e downloads não são usados nesta etapa.


In [0]:
# Objetivo:
# Verificar as regras de avaliação com exemplos controlados.

# Justificativa:
# Detectar métricas calculadas indevidamente sem rótulos e erros na persistência.

# Ação:
# Executar testes simulados sem LLM ou banco vetorial.


import tempfile
pasta_teste = Path(tempfile.mkdtemp(prefix="olist05_test_"))
q_teste = [{"pergunta_id": "Q01", "tema": "teste", "pergunta": "Há atraso?", "status_esperado": "respondido",
            "referencia_confirmada": "1", "criterio_aceite": "Relato explícito"}]
doc_teste = {"documento_id": "doc-teste", "texto": "Entrega atrasou.", "distancia": 0.1, "metadados": {}}
resultado_teste = {"status": "respondido", "afirmacoes": [{"texto": "O relato descreve atraso.", "fontes": ["E1"]}],
    "fontes_utilizadas": [{"fonte": "E1", "documento_id": "doc-teste", "trecho": "Entrega atrasou.", "texto": "Entrega atrasou."}],
    "chamadas_llm": [{"uso": {"total_tokens": 10}}]}
m_teste = abrir_execucao(pasta_teste, {"modelo": "simulado"}, q_teste)
contagem_teste = []
def responder_teste(*args):
    contagem_teste.append(1)
    return resultado_teste
executar_lote(pasta_teste, m_teste, q_teste, lambda p, k: [doc_teste], lambda *a: None, responder_teste)
executar_lote(pasta_teste, m_teste, q_teste, lambda p, k: [doc_teste], lambda *a: None, responder_teste)
assert len(contagem_teste) == 1
r_teste = carregar_resultados(pasta_teste, m_teste)
t_teste = preparar_tabelas_revisao(r_teste)
dfs_teste = [v[0].copy() for v in t_teste.values()]
s_teste, p_teste, _ = calcular_metricas(r_teste, q_teste, *dfs_teste, ks=(3,))
assert s_teste["fidelidade_nas_afirmacoes_revisadas"] is None
assert p_teste.precision_at_k.isna().all()
dfs_teste[0].loc[:, "relevante"] = "1"
dfs_teste[1].loc[:, "sustentada"] = "1"
dfs_teste[2].loc[:, ["completa", "clara"]] = "1"
s_teste, p_teste, _ = calcular_metricas(r_teste, q_teste, *dfs_teste, ks=(3,))
assert abs(float(p_teste.precision_at_k.iloc[0]) - 1 / 3) < 1e-10
assert s_teste["fidelidade_nas_afirmacoes_revisadas"] == 1.0
assert s_teste["acuracia_status_nas_referencias_executadas"] == 1.0
try:
    abrir_execucao(pasta_teste, {"modelo": "alterado"}, q_teste)
except ValueError:
    pass
else:
    raise AssertionError("Rodada deveria rejeitar configuração diferente.")
print("Testes sintéticos aprovados. Nenhuma métrica real foi presumida.")


In [0]:
# Objetivo:
# Salvar tabelas e um resumo legível da avaliação.

# Justificativa:
# Preparar evidências auditáveis para o repositório e para a demonstração.

# Ação:
# Persistir somente na rodada selecionada, mantendo os resultados brutos intactos.


if manifesto is None:
    print("Não há rodada real selecionada. Relatório real não será gravado.")
else:
    gravar_json(PASTA_EXECUCAO / "metricas.json", metricas)
    auditar_citacoes(registros).to_csv(PASTA_EXECUCAO / "auditoria_citacoes.csv", index=False, encoding="utf-8-sig")
    precisao_por_pergunta.to_csv(PASTA_EXECUCAO / "precision_por_pergunta.csv", index=False, encoding="utf-8-sig")
    comparacao_status.to_csv(PASTA_EXECUCAO / "comparacao_status.csv", index=False, encoding="utf-8-sig")
    df_execucoes.to_csv(PASTA_EXECUCAO / "resumo_execucoes.csv", index=False, encoding="utf-8-sig")
    resumo_md = "# Avaliação Databricks do RAG\n\n"
    resumo_md += f"Rodada: {EXECUCAO_ID}\n\nEstado: {metricas['estado_avaliacao']}\n\n"
    resumo_md += f"Perguntas executadas: {metricas['perguntas_executadas']} de {metricas['perguntas_planejadas']}.\n\n"
    resumo_md += "## Métricas e denominadores\n\n```json\n" + json.dumps(metricas, ensure_ascii=False, indent=2) + "\n```\n\n"
    resumo_md += "## Interpretação\n\nMétricas nulas indicam ausência de revisão ou referência suficiente; não representam zero. "
    resumo_md += "Revisão preenchida não significa qualidade aprovada. Consulte erros, falhas de validação e afirmações sem suporte. "
    resumo_md += "Não foram estimados recall, preço ou aprovação automática da solução.\n"
    (PASTA_EXECUCAO / "resumo_avaliacao.md").write_text(resumo_md, encoding="utf-8")
    print("Relatórios:", PASTA_EXECUCAO)

## Conclusão e próximos passos

### Ordem prática de utilização
1. Execute a preparação e revise o CSV de perguntas antes da primeira rodada real.
2. Configure o mesmo modelo do 04 e habilite EXECUTAR_AVALIACAO; comece com três perguntas.
3. Guarde o EXECUCAO_ID e retome até completar o conjunto.
4. Preencha as revisões de relevância, sustentação, completude e clareza.
5. Com EXECUTAR_AVALIACAO=False e o mesmo ID, reabra a rodada e recalcule as métricas sem consultar o LLM.
6. Investigue falhas; qualquer mudança no pipeline/modelo deve ser avaliada em outra rodada, preservando a anterior.

### Critérios de conclusão
- Todas as perguntas planejadas possuem uma tentativa registrada e referências confirmadas previamente.
- As revisões humanas estão preenchidas e os denominadores aparecem nos resultados.
- Erros técnicos, validacao_falhou e afirmações sem suporte foram analisados e documentados.
- Há exemplos revisados de resposta com fontes, insuficiência de evidência e fora de escopo.
- A equipe estabeleceu critérios de qualidade adequados ao uso e registrou as limitações.

Não foi inventado um limiar de aprovação (por exemplo, precision de 80%). Defina os critérios antes
de comparar melhorias e considere o tamanho reduzido e a composição deste conjunto de teste.
Um resultado alto em poucas perguntas não comprova generalização; perguntas elaboradas pela própria equipe
podem favorecer seu sistema. Amplie a avaliação e mantenha um conjunto separado para comparação final.

### Continuidade da entrega
Repita a avaliação com o ambiente e o LLM escolhidos no Databricks. Compare qualidade e cobertura
por tema, identificando diferenças de modelo, dados, prompts e hardware. Não misture os resultados em uma única média.
Depois, consolide README, arquitetura, dependências, exemplos e reprodução no GitHub e prepare o vídeo de até 7 minutos.
Arquivos de avaliação podem conter comentários completos e saídas do modelo: selecione os exemplos relevantes
para a entrega. O código será versionado no Git; os relatórios completos podem permanecer no armazenamento do projeto.
